## 1. Initialize dataset

In [1]:
from pathlib import Path

import polars as pl

from src.data.bag_of_words import BagOfWordsDatasetConfig, canonical_bags
from src.data.dataloading import DataloadingConfig
from src.data.parquet import TokenizedParquetDatasetConfig

/home/nlyu/Code/maxrl-statistics/src/data/parquet.py:10: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
dataset_folder = Path("artifacts/bow/dev")
bow_config = BagOfWordsDatasetConfig.init_or_load_from(
    folder=dataset_folder,
    snr=0.2,
    num_train_samples=100_000,
    num_val_samples=100_000,
    prompt_length=64,
    word_assignments=list(canonical_bags[7]),
    word_decay_power=1.0,
)
print(f"R² = {bow_config.rsq:.4f}")
display(bow_config.visualize())
pl.read_parquet(dataset_folder / "train.parquet").head(3)

R² = 0.0385


prompt,target,signal
str,f64,f64
"""okay good awful great bad awfu…",0.481098,0.226676
"""terrible okay bad good okay te…",0.938416,-0.356206
"""okay awful okay okay wonderful…",1.042304,0.010794


## 2. Initialize ParquetDataset

In [6]:
parquet_config = TokenizedParquetDatasetConfig(
    tokenizer_model_name="Qwen/Qwen2.5-0.5B",
    folder=dataset_folder,
    filter_samples_above_n_tokens=256,
    pad_to_multiple=8,
)
display(parquet_config.visualize())
ds = parquet_config.init_or_load_dataset()
print(f"Train: {ds.num_train_samples} * {ds.ceil_padded_seqlen}")
print(f"Val:   {ds.num_val_samples}   * {ds.ceil_padded_seqlen}")

Train: 73719 * 128
Val:   73605   * 128


## 3. Initialize dataloader

In [ ]:
dl_config = DataloadingConfig(
    train_batch_size=64,
    eval_batch_size=256,
    drop_last=True,
    world_size=1,
    rank=0,
)
train_dl = dl_config.get_train_dataloader(ds)
val_dl = dl_config.get_val_dataloader(ds)
tokens, targets, ground_truth = next(iter(train_dl))
print(f"tokens: {tuple(tokens.shape)}, targets: {tuple(targets.shape)}")